In [1]:
from pathlib import Path 
import os 
import re
from tqdm import tqdm 
import fitz  
import json 

Parsing data

In [2]:
def parse_pdf(file_path):
    """Повертає список (page_num, text) для кожної сторінки."""
    doc = fitz.open(file_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()
        if text.strip():        
            pages.append((page_num, text))
    doc.close()
    return pages



Chunk split

In [3]:
def chunk_text(text, chunk_size=1000, overlap=200):
    """Розбиває текст на чанки з перекриттям."""
    if not text or len(text) < 100:
        return []

    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if chunk.strip():
            chunks.append(chunk)

    # Якщо останній чанк занадто малий — злити з попереднім
    if len(chunks) > 1 and len(chunks[-1]) < 50:
        chunks[-2] += " " + chunks[-1]
        chunks.pop()

    return chunks

Create metada based on chunks

In [4]:
pdf_dir = Path("f:/General_Directory_VScode/Mini_RAG_assistant/downloaded_docs")
pdf_files = list(pdf_dir.glob("*.pdf"))

all_chunks = []
chunk_metadata = []
chunk_global_id = 0

for pdf_file in tqdm(pdf_files, desc="Обробка PDF"):
    pages = parse_pdf(pdf_file)          # [(page_num, text), ...]

    for page_num, page_text in pages:
        page_chunks = chunk_text(page_text, chunk_size=1000, overlap=200)

        for local_id, chunk in enumerate(page_chunks):
            all_chunks.append(chunk)
            chunk_metadata.append({
                "chunk_id":   chunk_global_id,   
                "local_id":   local_id,        
                "source":     pdf_file.name,      
                "page":       page_num,          
                "char_count": len(chunk),
            })
            chunk_global_id += 1

print(f"Файлів: {len(pdf_files)} | Чанків: {len(all_chunks)}")



Обробка PDF: 100%|██████████| 50/50 [00:13<00:00,  3.81it/s]

Файлів: 50 | Чанків: 26409


Save and check metadata

In [5]:
output = {"chunks": all_chunks, "metadata": chunk_metadata}
save_dir = "f:/General_Directory_VScode/Mini_RAG_assistant/save_data/chunks_with_metadata.json"

with open(save_dir, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Збережено у {save_dir}.json")

 
with open(save_dir, "r", encoding="utf-8") as f:
    data = json.load(f)

all_chunks   = data["chunks"]
chunk_metadata = data["metadata"]
print(f"Завантажено {len(all_chunks)} чанків")

Збережено у f:/General_Directory_VScode/Mini_RAG_assistant/save_data/chunks_with_metadata.json.json
Завантажено 26409 чанків


Building vector index

In [6]:
import chromadb
from chromadb.utils import embedding_functions


In [7]:
# 1. Завантажуємо чанки
with open(save_dir, 'r', encoding='utf-8') as f:
    data = json.load(f)
    chunks = data['chunks']
    metadata = data['metadata']

# 2. Створюємо клієнта
client = chromadb.PersistentClient(path="./chroma_db")

# 3. Створюємо колекцію
collection = client.get_or_create_collection(
    name="my_docs",
    embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# 4. Додаємо чанки в індекс
  
batch_size = 1000
total_chunks = len(chunks)
print(f"Додаємо {total_chunks} чанків порціями по {batch_size}...")

for i in range(0, total_chunks, batch_size):
    end = min(i + batch_size, total_chunks)

    batch_ids = [str(j) for j in range(i, end)]
    batch_chunks = chunks[i:end]
    batch_metadata = metadata[i:end]

    collection.add(
        ids=batch_ids,
        documents=batch_chunks,
        metadatas=batch_metadata
    )

    print(f"  ✅ Додано чанки {i+1}-{end} з {total_chunks}")


print(f"✅ Індекс створено! {len(chunks)} чанків")


Додаємо 26409 чанків порціями по 1000...
  ✅ Додано чанки 1-1000 з 26409
  ✅ Додано чанки 1001-2000 з 26409
  ✅ Додано чанки 2001-3000 з 26409
  ✅ Додано чанки 3001-4000 з 26409
  ✅ Додано чанки 4001-5000 з 26409
  ✅ Додано чанки 5001-6000 з 26409
  ✅ Додано чанки 6001-7000 з 26409
  ✅ Додано чанки 7001-8000 з 26409
  ✅ Додано чанки 8001-9000 з 26409
  ✅ Додано чанки 9001-10000 з 26409
  ✅ Додано чанки 10001-11000 з 26409
  ✅ Додано чанки 11001-12000 з 26409
  ✅ Додано чанки 12001-13000 з 26409
  ✅ Додано чанки 13001-14000 з 26409
  ✅ Додано чанки 14001-15000 з 26409
  ✅ Додано чанки 15001-16000 з 26409
  ✅ Додано чанки 16001-17000 з 26409
  ✅ Додано чанки 17001-18000 з 26409
  ✅ Додано чанки 18001-19000 з 26409
  ✅ Додано чанки 19001-20000 з 26409
  ✅ Додано чанки 20001-21000 з 26409
  ✅ Додано чанки 21001-22000 з 26409
  ✅ Додано чанки 22001-23000 з 26409
  ✅ Додано чанки 23001-24000 з 26409
  ✅ Додано чанки 24001-25000 з 26409
  ✅ Додано чанки 25001-26000 з 26409
  ✅ Додано чанки 26

Шукання схожиш векторів по запиті (який перетворюється в вектор для пошуку)

In [9]:
query = "NLP what is that"
results = collection.query(
    query_texts=[query],
    n_results=3
)
 
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"{i+1}. {meta['source']} (стор. {meta['page']}): {doc[:100]}...")

1. rf_aiinassetmanagement_practitioner-briefs_07_naturallanguageprocessing_online.pdf (стор. 1): s never 
been more important to understand:
● 
Explosion of text data: Filings, calls, and 
unstruct...
2. 2025.unlp-1.18.pdf (стор. 10): References
2024. Nlp-progress: repository to track the progress
in natural language processing (nlp)...
3. rf_aiinassetmanagement_practitioner-briefs_07_naturallanguageprocessing_online.pdf (стор. 1): actical guidance: who should read it, why it 
matters now, and how practitioners can use 
NLP and LL...
